<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring

This notebook uses the FlyRank Internship Warehouse dataset and focuses on March 2026 for development.

## Setup

The Hugging Face token is stored securely in Colab Secrets as `HF_TOKEN`.

In [1]:
!pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{REL}/fact_content_daily_performance/**/*.parquet"

print("Connected to FlyRank Internship Warehouse")

Connected to FlyRank Internship Warehouse


## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one pseudonymized client, one pseudonymized content item, and one report date.

**Table:** `fact_content_daily_performance`

**Time window:** March 2026 is used for development.

**Decision supported:** Rank content items for human review based on observed performance and refresh opportunity.

**Excluded:** Client and content IDs are excluded as model features because they are identifiers rather than generalizable signals.

## 2. Fields: feature / label / context / excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`

**Label / proxy:** A future performance outcome will be created later and will not be used as a feature.

**Context:** `report_date`, `month`, and data availability flags.

**Excluded:** Hash IDs and signals outside the initial five-feature frame.

## 3. Verify it with queries

The three queries below verify the grain, the March slice, and data availability.

In [2]:
# Query 1 — Verify the grain
query = f'''
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_keys
FROM read_parquet('{TABLE}', hive_partitioning=1)
WHERE month = '2026-03'
'''
con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_keys
0,9841378,9841378


In [3]:
# Query 2 — Row count and date span
query = f'''
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{TABLE}', hive_partitioning=1)
WHERE month = '2026-03'
'''
con.sql(query).df()

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [4]:
# Query 3 — Availability check
query = f'''
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet('{TABLE}', hive_partitioning=1)
WHERE month = '2026-03'
'''
con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0


### Five-feature frame

Each feature is knowable at the decision moment because it comes from historical performance observed before a future decision period.

- **Impressions:** historical search visibility.
- **Clicks:** historical search performance.
- **Average position:** historical ranking observations.
- **Sessions:** historical analytics data when GA4 is available.
- **Engaged sessions:** historical engagement data when GA4 is available.

In [5]:
# Build the five-feature frame
query = f'''
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{TABLE}', hive_partitioning=1)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
'''

df = con.sql(query).df()
print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 364347


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,0


### Deliberate leakage experiment

A label-derived feature is added on purpose to demonstrate leakage. The resulting score is artificially optimistic, so the feature is removed from the honest model.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

sample = df.dropna().sample(min(50000, len(df)), random_state=42).copy()
sample["label"] = (sample["gsc_clicks"] > sample["gsc_clicks"].median()).astype(int)

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X_train, X_test, y_train, y_test = train_test_split(
    sample[features], sample["label"], test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest ROC-AUC:", round(honest_auc, 3))

# Deliberate leakage: direct copy of the label
sample["leak_feature"] = sample["label"]

X_train, X_test, y_train, y_test = train_test_split(
    sample[features + ["leak_feature"]],
    sample["label"], test_size=0.2, random_state=42
)

leak_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
leak_model.fit(X_train, y_train)

leak_auc = roc_auc_score(y_test, leak_model.predict_proba(X_test)[:, 1])
print("Leaky ROC-AUC:", round(leak_auc, 3))
print("The leak feature is not used in the final feature set.")

Honest ROC-AUC: 0.743
Leaky ROC-AUC: 1.0
The leak feature is not used in the final feature set.


## 4. Data limits

- Client history is unbalanced.
- Some rows may have GSC data without GA4 data.
- March 2026 is used for development; the final month should remain separate from label development.
- This work supports prioritization and does not prove that refreshing content causes recovery.

## Self-check

- [x] Contract answers are included.
- [x] Three verification queries are included.
- [x] Availability uses `IS TRUE`.
- [x] Five features are documented.
- [x] A leakage experiment is included and removed.
- [x] Data limitations are stated.
- [ ] Run **Runtime → Run all** before committing.